In [ ]:
# ensure connection to cluster
spark

In [ ]:
# load packages
from pyspark.sql import SparkSession
import pandas as pd


# configure spark session to cluster specs
spark = SparkSession.builder \
    .appName("GTEx Full Load") \
    .config("spark.driver.memory", "24g") \
    .config("spark.executor.memory", "24g") \
    .config("spark.sql.parquet.mergeSchema", "false") \
    .config("spark.sql.parquet.filterPushdown", "true") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.maxResultSize", "8g") \
    .getOrCreate()

# Read data (~ 4 GB)
df = spark.read.parquet("gs://gene_datasets/GTEx_tissue_expression.parquet")
print((df.count(), len(df.columns)))

26/05/29 01:36:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
26/05/29 01:36:25 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:36:40 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:36:55 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:37:10 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:37:25 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:37:40 WA

In [ ]:
# Read sample attributes to link codes to tissues
attrs = pd.read_csv(
    "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
    sep="\t",
    index_col="SAMPID"
)

print(attrs.shape)
print(attrs["SMTSD"].value_counts())

(48231, 118)
Whole Blood                                              4369
Muscle - Skeletal                                        2352
Thyroid                                                  2033
Skin - Sun Exposed (Lower leg)                           1922
Lung                                                     1892
                                                         ... 
Pancreas - Acini                                            9
Small Intestine - Terminal Ileum - Lymphoid Aggregate       9
Colon - Transverse - Mucosa                                 9
Pancreas - Islets                                           3
Liver - Portal Tract                                        2
Name: SMTSD, Length: 70, dtype: int64


/tmp/ipykernel_26496/4006743547.py:3: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  attrs = pd.read_csv(


26/05/29 01:33:25 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:33:40 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:33:55 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:34:10 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:34:25 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:34:40 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registere

In [ ]:
# 1. Get only the SMTSD column (tissue label) — that's all we need
tissue_map = attrs["SMTSD"]

# 2. Find which sample columns in your TPM df have a known tissue label
#    (some samples in attrs may not be in your TPM file and vice versa)
sample_cols = [c for c in df.columns if c != "Description"]
matched = [c for c in sample_cols if c in tissue_map.index]
print(f"Samples in TPM file:        {len(sample_cols):,}")
print(f"Samples matched to tissue:  {len(matched):,}")
print(f"Unmatched:                  {len(sample_cols) - len(matched):,}")

Samples in TPM file:        4
Samples matched to tissue:  4
Unmatched:                  0


26/05/29 01:35:25 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:35:40 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:35:55 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/29 01:36:10 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
